# Tool Calling Agent

Tool Calling Agent란 **거대 언어 모델(LLM)**을 핵심 의사 결정 엔진으로 사용하여, 사전에 정의된 외부 도구(Tools)를 자율적으로 호출하고 그 결과를 활용하여 복잡한 문제를 해결하거나 질문에 답하는 인공지능 시스템

Tool Calling Agent의 기반은 LLM의 함수 호출(Function Calling) 능력!!

- 1. 도구 정의 및 등록: 개발자가 에이전트에게 제공할 수 있는 함수나 도구를 정의하고, 각 도구의 **목적(description)**과 **필요한 입력 매개변수(JSON Schema)**를 명시합니다. (함수 이름이 명확해야 하며, docstring이 도구 목적 잘 설명해야 함. 또한, 매개변수 타입 힌트가 있어야함 -> json 형태로 schema 정보 자동 생성함!)  
- 2. LLM의 의사 결정: 사용자 질문이 들어오면, LLM은 자신의 내부 지식과 등록된 도구의 설명을 비교하여 *'이 질문에 답하려면 어떤 도구를, 어떤 인수를 사용해서 호출해야 하는가?'*를 결정합니다.
- 3. Tool Call 생성: LLM은 답변 텍스트를 생성하는 대신, 호출해야 할 함수의 이름과 인수가 포함된 구조화된 JSON 객체(Tool Call)를 출력합니다.
- 4. 도구 실행: 에이전트 시스템(LangChain, LangGraph 등)은 LLM이 생성한 Tool Call을 읽고, 해당 함수를 실제 Python 환경에서 실행합니다.
- 5. 결과 피드백: 도구 실행 결과(Output)는 다시 **메시지 형태(ToolMessage)**로 LLM에게 전달되어 컨텍스트로 추가됩니다.
- 6. 최종 답변: LLM은 원래 질문, 자신이 내렸던 도구 호출 요청, 그리고 도구 실행 결과를 종합하여 최종적이고 정확한 답변을 생성합니다.

### 역할

>LLM (Language Model) |	에이전트의 두뇌 역할을 합니다. 도구 사용 여부와 사용할 도구의 인수를 결정하는 의사 결정자입니다.	

>Tools (도구) |	에이전트가 접근할 수 있는 외부 기능입니다. 이름, 설명, 스키마가 명확하게 정의되어야 합니다.	

>Parser (파서) |	LLM이 생성한 비표준 텍스트 또는 JSON 형태의 Tool Call 출력을 읽어 유효한 함수 호출 요청으로 변환합니다 : LangChain의 PydanticOutputParser 또는 자체 정의된 파싱 로직.

>Executor (실행기) |	파서가 추출한 함수 호출 요청을 받아, 실제 Python 함수를 실행하고 그 결과를 수집합니다 : LangGraph의 ToolExecutor, LangChain의 AgentExecutor.

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor
from langchain_classic.tools.retriever import create_retriever_tool
from typing import List, Dict, Any
from langchain_core.tools import tool

# --- 0. API 키 설정 (반드시 자신의 키로 대체하세요) ---
# os.environ["GOOGLE_API_KEY"] = "YOUR_GEMINI_API_KEY"

# --- 1. RAG(내부 지식)를 위한 데이터 및 리트리버 설정 ---

# 1.1. 예시 내부 문서 데이터
rag_docs = [
    Document(page_content="2025년 1분기 영업 목표는 스마트폰 판매량을 30% 증가시키는 것입니다.", metadata={"source": "Internal_Report_Q1_2025"}),
    Document(page_content="신규 직원 교육 프로그램은 12월 1일에 시작되며, 교육 장소는 본사 5층 대강당입니다.", metadata={"source": "HR_Memo"}),
]

# 1.2. VectorStore 및 Retriever 생성 (Chroma 사용)
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma.from_documents(rag_docs, embedding)
retriever = vectorstore.as_retriever()

# 1.3. Retrieval Tool 생성 (에이전트에게 제공할 도구)
retrieval_tool = create_retriever_tool(
    retriever,
    "internal_knowledge_retriever",
    "이 도구는 회사 내부의 2025년 목표나 직원 관련 정보에 대한 질문에 답할 때 사용합니다."
)
# retrival에 대한 간략한 설명이 다음처럼 반드시 들어가야 한다
# create_retriever_tool() 자체가 retriever를 LangChain Tool 객체로 감싸서 만들어주는 함수라서 @tool을 따로 붙이지 않는다. 

# --- 2. Google Search 도구 설정 (외부 검색) ---

# 에이전트가 사용할 수 있도록 LangChain Tool 형식으로 Google Search를 래핑합니다.
# 실제 실행 환경에서는 이 객체 대신 통합된 Google Search Tool이 자동으로 사용됩니다.


# @tool 데코레이터를 사용하여 LangChain Tool 형식으로 정의 (실제로는 API 통합 사용)
@tool
def google_search(query: str) -> str:
    """최신 뉴스, 실시간 정보, 또는 내부 문서에 없는 일반적인 지식에 대한 질문에 답할 때 웹 검색을 수행합니다."""
   
    """Search Google News by input keyword"""
    news_tool = GoogleNews()
    return news_tool.search_by_keyword(query, k=5)

# 도구 생성
@tool
def python_repl_tool(
    code: Annotated[str, "The python code to execute to generate your chart."],
):
    """Use this to execute python code. If you want to see the output of a value,
    you should print it out with `print(...)`. This is visible to the user."""
    result = ""
    try:
        result = PythonREPL().run(code)
    except BaseException as e:
        print(f"Failed to execute. Error: {repr(e)}")
    finally:
        return result
    
    
# --- 3. Agent 설정 및 실행 ---

# 3.1. LLM 및 도구 목록
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)
tools = [retrieval_tool, google_search, python_repl_tool] # tool들을 묶고 아래에 tool 역할을 부여한다. 

# 3.2. 프롬프트 로드 (Agent의 작동 방식을 정의)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            
            """
            You are a helpful agent with tools.
            당신은 문제 해결을 위해 Tool 들을 사용하여 최종 답변을 도출하는 유능한 AI 에이전트입니다.
            
            ## Tool information : 
            Make sure to use the `search_news` tool for searching keyword related news
            Make sure to use the `python_repl_tool` tool for when python code execute   
            Make sure to use the `retrieval_tool` tool for when internal_knowledge_retriever 
            주어진 질문에 대해서 어떤 Tool 을 호출해서 답변할지, 단계별로 생각해서 답변(step-by-step)해줘.
            
            """
            
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
) # 순서가 중요하다. 이 순서대로 LLM에게 전달한다. 
# placeholder", "{agent_scratchpad}에는 에이전트의 도구 호출 요청, 실행 결과 등 에이전트가 이전에 어떤 도구를 호출했고 어떤 결과를 받았는지 기억하는 부분이다. 

#   1. system: 너는 어떤 역할이고 어떤 규칙을 따라야 하는지 알려줌
#   2. chat_history: 이전 대화 기록을 넣음
#   3. human: 현재 사용자 질문을 넣음
#   4. agent_scratchpad: 에이전트의 도구 호출 중간 기록을 넣음



# 3.3. Agent 생성 (Tool Calling 방식 사용)
agent = create_tool_calling_agent(llm, tools, prompt)
agent

이렇게 묶어주고 tool을 주고 나면 AgentExector라는 클래스가 있어야 agent를 실행할  수 있다. 

In [ ]:
# 3.4. Agent Executor 생성 (Agent 실행기)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    max_execution_time=10,
    handle_parsing_errors=True,
)
# ----------------------------------------------------
# 4. Agent RAG 실행 및 결과
# ----------------------------------------------------

print("--- 🔍 내부 RAG 지식 질문 (retriever 사용 예상) ---")
internal_query = "2025년 1분기 영업 목표는 무엇인가요?"

# Agent Executor는 'internal_knowledge_retriever'를 선택합니다.
print(f"사용자 질문: {internal_query}")
result_internal = agent_executor.invoke({"input": internal_query})
print(f"\n✅ 최종 답변: {result_internal['output']}\n")
# 내가 어떤 tool을 호출하라고 한게 아니라, 질문이 오면 질문을 보고 자동적으로 어떤 tool을 쓸지 결정을 스스로 해서 결과를 반환한 것을 볼 수 있다. 

In [ ]:
print("-" * 50)

print("--- 🌐 외부 검색 질문 (google_search 사용 예상) ---")
external_query = "AI 투자와 관련된 뉴스를 검색해 주고, 검색된 결과를 바탕으로 간단하고 전략적으로 요약해줘"

# Agent Executor는 'google_search'를 선택합니다.
print(f"사용자 질문: {external_query}")
result_external = agent_executor.invoke({"input": external_query})
print(f"\n✅ 최종 답변: {result_external['output']}\n")
# 마찬가지로 tool 호출하라고 지정하지는 않았지만, 다음처럼 스스로 tool을 선택했다

결국 내가 tool을 만들어서 llm에게 바인딩 할 수 있으니, 우리가 다양한 tool을 구현해서 agent를 만들 수 있다. 스스로 판단한다!


> 요즘 방법론에 맞는 방식은 이렇게 프롬프트를 다르게 지정해 주는 방식으로 구현하는 것이라고 한다. 